# Step 2 — TOD Stop and Access Point Station Assignment

Reads the manually curated station, stop, and pedestrian access point datasets
from the project geodatabase, then spatially assigns each stop and access point
to its parent station using a progressive buffer analysis (150 ft → 300 ft → 500 ft → 1000 ft).

Outputs development layers (`tod_stations_dev`, `tod_stops_dev`,
`tod_access_points_dev`) to the shared GeoPackage and writes a review Excel
workbook (`SB79_tod_review.xlsx`) for manual resolution before the final
re-integration step.

> **Pipeline order:** Run after `1_gtfs_tod_stop_classification.ipynb`.
> After running this notebook:
> 1. Open `SB79_tod_review.xlsx`.
> 2. For any stop or access point that needs a corrected station assignment,
>    set `assignment_status` = `manual_station_assignment` and enter the
>    correct `station_id`.  The dropdown only allows `conflict`, `no_match`,
>    and `manual_station_assignment`.
> 3. Save the workbook, then continue in
>    `3_tod_station_assignment_reintegration.ipynb`.
>
> All data source paths are defined in `config.py`.

In [23]:
import pandas as pd
import geopandas as gpd
import uuid
from pathlib import Path
from typing import Optional
from config import *

In [24]:
# All paths and layer names are imported from config.py
tod_db_gdb_path = TOD_DATABASE_GDB
tod_gtfs_db_gdb_path = TOD_DATABASE_GPKG

In [25]:
# Read stations and stops from curated geodatabase
stations_gdf = gpd.read_file(tod_db_gdb_path, layer=GDB_STATIONS_LAYER)
print(f"Loaded {len(stations_gdf)} stations from geodatabase")

stops_gdf = gpd.read_file(tod_db_gdb_path, layer=GDB_STOPS_LAYER)
print(f"Loaded {len(stops_gdf)} stops from geodatabase")

# Read station override list — stations in this list are excluded from spatial
# assignment so that stops and access points cannot be assigned to non-TOD stations.
station_overrides_df = pd.read_excel(STATIONS_OVERRIDES_XLSX, sheet_name=STATION_OVERRIDES_SHEET)
print(f"Loaded {len(station_overrides_df)} station overrides from {STATIONS_OVERRIDES_XLSX.name}")

Loaded 607 stations from geodatabase
Loaded 11357 stops from geodatabase
Loaded 157 station overrides from 2026_03_04_tod_stations_overrides.xlsx


In [26]:
stations_gdf

,station_id,station_name,location_type,manually_added,geometry
0,mtc:powell,Powell,1,NaN,POINT (-122.40737 37.78459)
1,mtc:fruitvale,Fruitvale,1,NaN,POINT (-122.22412 37.77486)
2,mtc:warm-springs-south-fremont-bart,Warm Springs South Fremont BART,1,NaN,POINT (-121.93925 37.50199)
3,mtc:caltrain-4th-&-king,Caltrain 4th & King,1,NaN,POINT (-122.39487 37.77649)
4,mtc:el-cerrito-del-norte-bart,El Cerrito Del Norte BART,1,NaN,POINT (-122.31699 37.92529)
...,...,...,...,...,...
602,mtc:saint-james-eb,Saint James Station EB,1,1.0,POINT (-121.89218 37.33842)
603,mtc:saint-james-wb,Saint James Station WB,1,1.0,POINT (-121.891 37.33855)
604,mtc:santa-clara-wb,Santa Clara WB,1,1.0,POINT (-121.88936 37.33631)
605,mtc:san-antonio-wb,San Antonio WB,1,1.0,POINT (-121.88729 37.33353)


## Per-agency pedestrian access point sources

Read each agency's access point file as a separate GeoDataFrame so schemas
can be inspected before building the merge.  Source paths are defined in
`config.py` under `ACCESS_PTS_SOURCES`.

In [4]:
# Read each agency's access point source as a separate GeoDataFrame
def _read_access_pts(
    path: Optional[Path | str],
    layer: Optional[str] = None,
) -> Optional[gpd.GeoDataFrame]:
    """Read a shapefile (zip) or GDB layer into a GeoDataFrame.

    Returns None if path is None.
    """
    if path is None:
        return None
    kwargs = {"layer": layer} if layer is not None else {}
    return gpd.read_file(path, **kwargs)

access_pts_ba_gdf = _read_access_pts(ACCESS_PTS_BA_PATH, ACCESS_PTS_BA_LAYER)
access_pts_ct_gdf = _read_access_pts(ACCESS_PTS_CT_PATH, ACCESS_PTS_CT_LAYER)
access_pts_ac_gdf = _read_access_pts(ACCESS_PTS_AC_PATH, ACCESS_PTS_AC_LAYER)
access_pts_sc_gdf = _read_access_pts(ACCESS_PTS_SC_PATH, ACCESS_PTS_SC_LAYER)
access_pts_sf_gdf = _read_access_pts(ACCESS_PTS_SF_PATH, ACCESS_PTS_SF_LAYER)

_agency_gdfs = {
    "BA": access_pts_ba_gdf,
    "CT": access_pts_ct_gdf,
    "AC": access_pts_ac_gdf,
    "SC": access_pts_sc_gdf,
    "SF": access_pts_sf_gdf,
}

for agency, gdf in _agency_gdfs.items():
    if gdf is None:
        print(f"{agency}: not loaded (path is None)")
    else:
        print(f"{agency}: {len(gdf)} rows | CRS: {gdf.crs} | columns: {list(gdf.columns)}")


BA: 183 rows | CRS: EPSG:26911 | columns: ['stop_id', 'stop_name', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'tts_stop_n', 'platform_c', 'location_t', 'parent_sta', 'stop_timez', 'wheelchair', 'level_id', 'route_info', 'geometry']
CT: 260 rows | CRS: EPSG:4326 | columns: ['stop_id', 'stop_name', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'tts_stop_n', 'platform_c', 'location_t', 'parent_sta', 'stop_timez', 'wheelchair', 'level_id', 'route_info', 'geometry']
AC: 92 rows | CRS: EPSG:4326 | columns: ['access_id', 'station_id', 'acces_poin', 'location_t', 'stop_lat', 'stop_lon', 'stop_name', 'geometry']
SC: 239 rows | CRS: EPSG:4326 | columns: ['stop_id', 'stop_name', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'location_t', 'parent_sta', 'level_id', 'route_info', 'vta_comm', 'geometry']
SF: 603 rows | CRS: EPSG:3857 | columns: ['stop_id_12', 'stop_name_', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_i

In [5]:
# Normalize each agency GDF to a common schema before merging.
#
# Generic rename (applied to all agencies, only if target doesn't already exist):
#   stop_id     → access_id
#   stop_name   → access_point_name
#   location_t  → location_type
#   parent_sta  → station_id
#
# Per-agency pre-renames fix truncated/variant column names before the
# generic map runs (e.g. shapefile 10-char field name truncation).
#
# After renaming, keep only the five canonical columns.

RENAME_MAP = {
    "stop_id":    "access_id",
    "stop_name":  "access_point_name",
    "location_t": "location_type",
    "parent_sta": "station_id",
}

# Agency-specific renames applied BEFORE RENAME_MAP.
# Add entries here whenever a source file uses non-standard column names.
AGENCY_PRE_RENAME = {
    "SF": {
        "stop_id_12": "stop_id",
        "stop_name_": "stop_name",
    },
}

KEEP_COLS = ["access_id", "access_point_name", "location_type", "station_id", "geometry"]


def normalize_access_pts(
    gdf: Optional[gpd.GeoDataFrame],
    agency: str,
) -> Optional[gpd.GeoDataFrame]:
    """Rename source columns to the canonical schema, then select KEEP_COLS.

    Applies any agency-specific pre-renames from AGENCY_PRE_RENAME before
    the generic RENAME_MAP, then subsets to KEEP_COLS. Returns None if gdf
    is None.
    """
    if gdf is None:
        return None
    gdf = gdf.copy()

    # Apply any agency-specific pre-renames first
    pre_renames = AGENCY_PRE_RENAME.get(agency, {})
    for src, tgt in pre_renames.items():
        if src in gdf.columns:
            gdf = gdf.rename(columns={src: tgt})
            print(f"  {agency}: pre-rename '{src}' → '{tgt}'")

    # Apply generic rename map
    for src, tgt in RENAME_MAP.items():
        if src in gdf.columns and tgt not in gdf.columns:
            gdf = gdf.rename(columns={src: tgt})
        elif src in gdf.columns and tgt in gdf.columns:
            print(f"  {agency}: '{tgt}' already exists — skipping rename of '{src}'")

    missing = [c for c in KEEP_COLS if c != "geometry" and c not in gdf.columns]
    if missing:
        print(f"  {agency}: ⚠️  columns not found after rename: {missing}")

    cols = [c for c in KEEP_COLS if c in gdf.columns]
    return gdf[cols].copy()


access_pts_ba_norm = normalize_access_pts(access_pts_ba_gdf, "BA")
access_pts_ct_norm = normalize_access_pts(access_pts_ct_gdf, "CT")
access_pts_ac_norm = normalize_access_pts(access_pts_ac_gdf, "AC")
access_pts_sc_norm = normalize_access_pts(access_pts_sc_gdf, "SC")
access_pts_sf_norm = normalize_access_pts(access_pts_sf_gdf, "SF")

_norm_gdfs = {
    "BA": access_pts_ba_norm,
    "CT": access_pts_ct_norm,
    "AC": access_pts_ac_norm,
    "SC": access_pts_sc_norm,
    "SF": access_pts_sf_norm,
}

for agency, gdf in _norm_gdfs.items():
    if gdf is None:
        print(f"{agency}: not loaded")
    else:
        print(f"{agency}: {len(gdf)} rows | columns: {list(gdf.columns)}")


  SF: pre-rename 'stop_id_12' → 'stop_id'
  SF: pre-rename 'stop_name_' → 'stop_name'
BA: 183 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
CT: 260 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
AC: 92 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
SC: 239 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
SF: 603 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']


## Merge and clean merged access points

Concat all normalized agency GDFs into a single `access_pts_gdf`, then apply
deduplication and access_id gap-filling before spatial assignment.

In [6]:
# Concat all normalized agency GDFs into a single access_pts_gdf.
# Reproject each source to EPSG:4326 before concat to resolve CRS mismatches.
_to_concat = {k: v for k, v in _norm_gdfs.items() if v is not None}

for agency, gdf in _to_concat.items():
    print(f"  {agency}: {gdf.crs}")

access_pts_gdf = gpd.GeoDataFrame(
    pd.concat(
        [gdf.to_crs("EPSG:4326").assign(agency=agency) for agency, gdf in _to_concat.items()],
        ignore_index=True,
    ),
    crs="EPSG:4326",
)

print(f"\nMerged {len(_to_concat)} agency sources into {len(access_pts_gdf)} total access points")
for agency, gdf in _to_concat.items():
    print(f"  {agency}: {len(gdf)} rows")

  BA: EPSG:26911
  CT: EPSG:4326
  AC: EPSG:4326
  SC: EPSG:4326
  SF: EPSG:3857

Merged 5 agency sources into 1377 total access points
  BA: 183 rows
  CT: 260 rows
  AC: 92 rows
  SC: 239 rows
  SF: 603 rows


In [7]:
import shapely

def _stable_access_id(geom) -> str:
    """Generate a stable, human-readable access_id from a WGS-84 point geometry.

    Uses lat/lon rounded to 5 decimal places (~1 m precision) so the ID is
    reproducible across re-runs as long as the source coordinates don't change.
    Format: ``geom:<lat>,<lon>``  e.g. ``geom:37.77891,-122.41234``
    """
    return f"geom:{geom.y:.5f},{geom.x:.5f}"


# ── Step 0: snap coordinates to 5-decimal-place grid ──────────────────────────
# Different agency exports sometimes store the same physical point with tiny
# floating-point differences (e.g. -122.412340001 vs -122.41234).  Snapping to
# a 1e-5 degree grid (~1 m) normalises those before any dedup comparison.
access_pts_gdf["geometry"] = shapely.set_precision(
    access_pts_gdf.geometry.values, grid_size=1e-5
)
print(f"Snapped {len(access_pts_gdf)} geometries to 5-decimal-place grid (~1 m)")

# ── Step 1: drop exact (access_id, geometry) duplicates ───────────────────────
# Same point submitted twice from the same agency with the same ID.
before = len(access_pts_gdf)
access_pts_gdf = access_pts_gdf.drop_duplicates(subset=["access_id", "geometry"], keep="first")
print(
    f"Dropped {before - len(access_pts_gdf)} exact (access_id + geometry) duplicates "
    f"({len(access_pts_gdf)} remaining)"
)

# ── Step 2: drop duplicate geometries ─────────────────────────────────────────
# Two access points at exactly the same location, regardless of access_id.
# There is no meaningful way to distinguish them spatially, so keep only the first.
before_geom = len(access_pts_gdf)
access_pts_gdf = access_pts_gdf[~access_pts_gdf.geometry.duplicated(keep="first")].copy()
print(
    f"Dropped {before_geom - len(access_pts_gdf)} duplicate geometry rows "
    f"({len(access_pts_gdf)} remaining)"
)

# ── Step 3: resolve any remaining access_id collisions ────────────────────────
# After the geometry dedup above each geometry is unique, so a coordinate-based
# stable ID is a safe replacement for any colliding access_ids.
dup_id_mask = access_pts_gdf.duplicated(subset=["access_id"], keep="first")
n_dup_ids = dup_id_mask.sum()
if n_dup_ids > 0:
    print(
        f"{n_dup_ids} rows share an access_id with a different geometry — reassigning stable IDs"
    )
    access_pts_gdf.loc[dup_id_mask, "access_id"] = [
        _stable_access_id(geom)
        for geom in access_pts_gdf.loc[dup_id_mask, "geometry"]
    ]
else:
    print("access_id is unique after dedup")

assert access_pts_gdf["access_id"].duplicated().sum() == 0, "access_id is still not unique!"


Snapped 1377 geometries to 5-decimal-place grid (~1 m)
Dropped 98 exact (access_id + geometry) duplicates (1279 remaining)
Dropped 97 duplicate geometry rows (1182 remaining)
604 rows share an access_id with a different geometry — reassigning stable IDs


In [8]:
# Assign a stable, coordinate-based access_id to any rows missing one.
# The generated ID uses the format ``auto:<lat>,<lon>`` so it is both
# human-readable and reproducible across re-runs (relies on _stable_access_id
# defined in the cell above).
# Catches both true nulls (NaN/None) and blank/whitespace-only strings,
# which isna() alone would miss.

blank_id_mask = access_pts_gdf["access_id"].isna() | (
    access_pts_gdf["access_id"].astype(str).str.strip() == ""
)
num_blank = blank_id_mask.sum()
print(f"Found {num_blank} rows with null or blank access_id")

if num_blank > 0:
    access_pts_gdf.loc[blank_id_mask, "access_id"] = [
        _stable_access_id(geom)
        for geom in access_pts_gdf.loc[blank_id_mask, "geometry"]
    ]
    print(f"Assigned coordinate-based stable access_id to {num_blank} rows")

# Confirm no remaining nulls, blanks, or duplicates
still_blank = (
    access_pts_gdf["access_id"].isna()
    | (access_pts_gdf["access_id"].astype(str).str.strip() == "")
).sum()
dup_count = access_pts_gdf["access_id"].duplicated().sum()
print(f"\nRemaining null/blank access_id: {still_blank}")
print(f"Remaining duplicate access_id:  {dup_count}")
print(f"Total access points:            {len(access_pts_gdf)}")


Found 2 rows with null or blank access_id
Assigned coordinate-based stable access_id to 2 rows

Remaining null/blank access_id: 0
Remaining duplicate access_id:  0
Total access points:            1182


In [9]:
# Join to the GTFS access_points layer written by Step 1 to assign station_id.
#
# station_id values from the per-agency source files are dropped before this
# join — they may be stale or use different IDs and cannot be trusted.
# The GTFS join is the sole authoritative source for initial station_id values;
# anything still unassigned after the join is handled by spatial assignment.

# Drop any station_id that came through from the per-agency source files
if "station_id" in access_pts_gdf.columns:
    n_source_assigned = access_pts_gdf["station_id"].notna().sum()
    print(f"Dropping {n_source_assigned} station_id value(s) from per-agency source data")
    access_pts_gdf = access_pts_gdf.drop(columns=["station_id"])

gtfs_access_pts_gdf = gpd.read_file(tod_gtfs_db_gdb_path, layer=GPKG_ACCESS_PTS_LAYER)
print(f"Loaded {len(gtfs_access_pts_gdf)} GTFS access points from GeoPackage (Step 1 output)")
print(f"  GTFS records with station_id: {gtfs_access_pts_gdf['station_id'].notna().sum()}")

access_pts_gdf = access_pts_gdf.merge(
    gtfs_access_pts_gdf[["access_id", "station_id"]],
    on="access_id",
    how="left",
)

n_assigned = access_pts_gdf["station_id"].notna().sum()
n_unassigned = access_pts_gdf["station_id"].isna().sum()
print(f"\nAfter GTFS station_id join:")
print(f"  Access points with station_id:    {n_assigned}")
print(f"  Access points without station_id: {n_unassigned} (will be resolved by spatial assignment)")

Dropping 1121 station_id value(s) from per-agency source data
Loaded 1236 GTFS access points from GeoPackage (Step 1 output)
  GTFS records with station_id: 1236

After GTFS station_id join:
  Access points with station_id:    113
  Access points without station_id: 1069 (will be resolved by spatial assignment)


In [10]:
# check if stops_gdf have any duplicate stop_id values
duplicate_stop_ids = stops_gdf[stops_gdf.duplicated(subset=["stop_id"], keep=False)]
print(f"Found {len(duplicate_stop_ids)} duplicate stop_id values in stops_gdf")
if len(duplicate_stop_ids) > 0:
    print("Sample duplicate stop_id values:")
    print(duplicate_stop_ids["stop_id"].unique()[:5].tolist())

Found 0 duplicate stop_id values in stops_gdf


In [11]:
# Check current CRS and prepare for spatial assignment
print("Current CRS:")
print(f"Stations: {stations_gdf.crs}")
print(f"Stops: {stops_gdf.crs}")
print(f"Access Points: {access_pts_gdf.crs}")

# Define target CRS for accurate distance measurements
# Using UTM Zone 10N NAD83 - EPSG:26910 (meters)
target_crs = "EPSG:26910"

print(f"\nProjecting to {target_crs} for accurate distance measurements in meters...")

Current CRS:
Stations: EPSG:4326
Stops: EPSG:4326
Access Points: EPSG:4326

Projecting to EPSG:26910 for accurate distance measurements in meters...


In [12]:
# Filter stops to remove any that are flagged for removal by station overrides
station_rm_list = station_overrides_df['station_id'].tolist()
before = len(stations_gdf)
stations_gdf = stations_gdf[~stations_gdf['station_id'].isin(station_rm_list)]
print(f"Removed {before - len(stations_gdf)} stations based on station overrides")

Removed 157 stations based on station overrides


In [13]:
# Filter stops to only include TOD stops (tod_stop = 1) for station assignment
print("Filtering stops to only include TOD-applicable stops...")
print(f"Total stops before filtering: {len(stops_gdf)}")

# Check if tod_stop column exists and filter
if "tod_stop" in stops_gdf.columns:
    tod_stops_gdf = stops_gdf[stops_gdf["tod_stop"] == 1].copy()
    print(f"TOD stops (tod_stop=1): {len(tod_stops_gdf)}")
else:
    print("Warning: 'tod_stop' column not found. Using all stops.")
    tod_stops_gdf = stops_gdf.copy()

print(f"Proceeding with {len(tod_stops_gdf)} stops for station assignment")

Filtering stops to only include TOD-applicable stops...
Total stops before filtering: 11357
TOD stops (tod_stop=1): 755
Proceeding with 755 stops for station assignment


In [14]:
# Project all datasets to target CRS for spatial operations
stations_proj = stations_gdf.to_crs(target_crs)
stops_proj = tod_stops_gdf.to_crs(target_crs)
access_pts_proj = access_pts_gdf.to_crs(target_crs)

print(f"Projected {len(stations_proj)} stations")
print(f"Projected {len(stops_proj)} TOD stops")
print(f"Projected {len(access_pts_proj)} access points")

Projected 450 stations
Projected 755 TOD stops
Projected 1182 access points


In [15]:
def assign_station_ids_spatially(
    points_gdf: gpd.GeoDataFrame,
    stations_gdf: gpd.GeoDataFrame,
    id_col: str,
    station_id_col: str = "station_id",
    max_distance: float = 304.8,
) -> tuple[gpd.GeoDataFrame, pd.DataFrame]:
    """Assign station IDs to points using progressive buffer distances.

    Processes unassigned points at increasing buffer sizes (45.7 m, 91.4 m,
    152.4 m, 304.8 m). Points with exactly one station match are assigned
    directly. Points matching multiple stations at the same distance are
    flagged as conflicts and excluded from further buffer passes.

    Args:
        points_gdf: Points to assign to stations. Must be in the same CRS
            as stations_gdf.
        stations_gdf: Station locations used to generate buffers.
        id_col: Column name for point IDs (e.g. ``"stop_id"``).
        station_id_col: Column name for station IDs to write into
            points_gdf. Created if it does not already exist.
            Defaults to ``"station_id"``.
        max_distance: Maximum buffer distance in metres. Processing stops
            after this distance is reached. Defaults to 304.8 (1000 ft).

    Returns:
        A two-tuple of:
        - Annotated points GeoDataFrame with ``station_id``,
          ``assignment_status`` (``"assigned"`` | ``"conflict"`` |
          ``"no_match"``), ``buffer_distance_m``, and
          ``station_id_corrected`` (``False`` by default; set to ``True``
          when a user manually changes the station_id after spatial
          assignment) columns.
        - DataFrame of per-buffer assignment statistics.
    """

    # Create working copy
    points = points_gdf.copy()

    # Initialize station_id column if it doesn't exist
    if station_id_col not in points.columns:
        points[station_id_col] = None

    # Seed assignment_status from any pre-existing station_id values (e.g.
    # from a GTFS join upstream).  Everything else starts as "no_match".
    points["assignment_status"] = points[station_id_col].apply(
        lambda x: "assigned" if pd.notna(x) else "no_match"
    )
    points["buffer_distance_m"] = pd.NA
    # False by default; set to True (here or downstream) whenever a user
    # manually changes the station_id from what spatial assignment produced.
    points["station_id_corrected"] = False

    assignment_stats = []

    # Progressive buffer distances in meters (converted from feet)
    # 150ft ≈ 45.7m, 300ft ≈ 91.4m, 500ft ≈ 152.4m, 1000ft ≈ 304.8m
    buffer_distances = [45.7, 91.4, 152.4, 304.8]

    for buffer_dist in buffer_distances:
        feet_equiv = buffer_dist * 3.28084  # Convert meters to feet for display
        print(f"\n--- Buffer distance: {buffer_dist:.1f}m ({feet_equiv:.0f}ft) ---")

        # Only process points still waiting for a match
        unassigned_mask = points["assignment_status"] == "no_match"
        unassigned_points = points[unassigned_mask].copy()

        if len(unassigned_points) == 0:
            print("All points assigned or in conflict!")
            break

        print(f"Processing {len(unassigned_points)} unassigned points")

        # Create buffers around stations
        station_buffers = stations_gdf.copy()
        station_buffers["geometry"] = stations_gdf.geometry.buffer(buffer_dist)

        # Spatial join to find intersections
        intersections = gpd.sjoin(
            unassigned_points[["geometry", id_col]],
            station_buffers[["geometry", "station_id"]],
            how="inner",
            predicate="intersects",
        ).drop(columns=["index_right"], errors="ignore")

        if len(intersections) == 0:
            print(f"No assignments at {buffer_dist:.1f}m")
            continue

        # Find points with single vs multiple station matches
        match_counts = intersections.groupby(id_col).size()
        single_matches = match_counts[match_counts == 1].index
        multiple_matches = match_counts[match_counts > 1].index

        # Assign single matches — write station_id + status directly into points
        for _, row in intersections[intersections[id_col].isin(single_matches)].iterrows():
            mask = points[id_col] == row[id_col]
            points.loc[mask, station_id_col] = row["station_id"]
            points.loc[mask, "assignment_status"] = "assigned"
            points.loc[mask, "buffer_distance_m"] = buffer_dist

        # Flag conflicts — mark status and record the distance; skip at larger buffers
        if len(multiple_matches) > 0:
            conflict_mask = points[id_col].isin(multiple_matches)
            points.loc[conflict_mask, "assignment_status"] = "conflict"
            points.loc[conflict_mask, "buffer_distance_m"] = buffer_dist

        assigned_count = len(single_matches)
        conflict_count = len(multiple_matches)
        print(f"Assigned: {assigned_count} | New conflicts: {conflict_count}")

        assignment_stats.append(
            {
                "buffer_distance_m": buffer_dist,
                "assigned": assigned_count,
                "conflicts": conflict_count,
                "remaining_unassigned": unassigned_mask.sum() - assigned_count - conflict_count,
            }
        )

        if buffer_dist >= max_distance:
            break

    # Final summary
    n_assigned = (points["assignment_status"] == "assigned").sum()
    n_conflict = (points["assignment_status"] == "conflict").sum()
    n_no_match = (points["assignment_status"] == "no_match").sum()
    print(f"\n=== FINAL RESULTS ===")
    print(f"Total points:  {len(points)}")
    print(f"  assigned:    {n_assigned}")
    print(f"  conflict:    {n_conflict}  ← needs review")
    print(f"  no_match:    {n_no_match}  ← needs review")

    return points, pd.DataFrame(assignment_stats)


In [16]:
# Apply spatial assignment to stops (only for stops without existing station_id)
print("=== ASSIGNING STATION IDs TO TOD STOPS ===")

stops_proj_result, stops_stats = assign_station_ids_spatially(
    stops_proj, stations_proj, id_col="stop_id", station_id_col="station_id", max_distance=304.8
)

print(f"\nTOD stops assignment statistics:")
print(stops_stats)


=== ASSIGNING STATION IDs TO TOD STOPS ===

--- Buffer distance: 45.7m (150ft) ---
Processing 505 unassigned points
Assigned: 449 | New conflicts: 19

--- Buffer distance: 91.4m (300ft) ---
Processing 37 unassigned points
Assigned: 31 | New conflicts: 3

--- Buffer distance: 152.4m (500ft) ---
Processing 3 unassigned points
Assigned: 2 | New conflicts: 0

--- Buffer distance: 304.8m (1000ft) ---
Processing 1 unassigned points
No assignments at 304.8m

=== FINAL RESULTS ===
Total points:  755
  assigned:    732
  conflict:    22  ← needs review
  no_match:    1  ← needs review

TOD stops assignment statistics:
   buffer_distance_m  assigned  conflicts  remaining_unassigned
0               45.7       449         19                    37
1               91.4        31          3                     3
2              152.4         2          0                     1


In [17]:
# Apply spatial assignment to access points
print("=== ASSIGNING STATION IDs TO ACCESS POINTS ===")

access_pts_proj_result, access_pts_stats = assign_station_ids_spatially(
    access_pts_proj,
    stations_proj,
    id_col="access_id",
    station_id_col="station_id",
    max_distance=304.8,
)

print(f"\nAccess points assignment statistics:")
print(access_pts_stats)


=== ASSIGNING STATION IDs TO ACCESS POINTS ===

--- Buffer distance: 45.7m (150ft) ---
Processing 1069 unassigned points
Assigned: 671 | New conflicts: 24

--- Buffer distance: 91.4m (300ft) ---
Processing 374 unassigned points
Assigned: 206 | New conflicts: 13

--- Buffer distance: 152.4m (500ft) ---
Processing 155 unassigned points
Assigned: 98 | New conflicts: 21

--- Buffer distance: 304.8m (1000ft) ---
Processing 36 unassigned points
Assigned: 24 | New conflicts: 8

=== FINAL RESULTS ===
Total points:  1182
  assigned:    1112
  conflict:    66  ← needs review
  no_match:    4  ← needs review

Access points assignment statistics:
   buffer_distance_m  assigned  conflicts  remaining_unassigned
0               45.7       671         24                   374
1               91.4       206         13                   155
2              152.4        98         21                    36
3              304.8        24          8                     4


In [18]:
# Rows still requiring review after spatial assignment

stops_proj_result[stops_proj_result["assignment_status"] != "assigned"][
    ["stop_id", "station_id", "assignment_status", "buffer_distance_m"]
].sort_values("assignment_status")

,stop_id,station_id,assignment_status,buffer_distance_m
312,13311,NaN,conflict,45.7
1101,17073,NaN,conflict,45.7
1102,18059,NaN,conflict,45.7
1640,17954,NaN,conflict,91.4
1658,14301,NaN,conflict,45.7
1948,14821,NaN,conflict,45.7
2112,15639,NaN,conflict,45.7
2124,15651,NaN,conflict,45.7
2133,15662,NaN,conflict,45.7
2134,15661,NaN,conflict,45.7


In [19]:
access_pts_proj_result[access_pts_proj_result["assignment_status"] != "assigned"][
    ["access_id", "station_id", "assignment_status", "buffer_distance_m"]
].sort_values("assignment_status")

,access_id,station_id,assignment_status,buffer_distance_m
99,"geom:37.33635,-121.89164",NaN,conflict,152.4
100,"geom:37.33650,-121.89018",NaN,conflict,91.4
366,"geom:37.79040,-122.39505",NaN,conflict,152.4
367,"geom:37.79064,-122.39534",NaN,conflict,152.4
368,"geom:37.79059,-122.39528",NaN,conflict,152.4
...,...,...,...,...
1181,"geom:37.77558,-122.41881",NaN,conflict,91.4
126,"geom:37.36454,-121.87316",NaN,no_match,<NA>
127,"geom:37.36579,-121.87355",NaN,no_match,<NA>
599,New_VTA_Eastridge_LRT_Station,NaN,no_match,<NA>


In [20]:
# Reproject results back to original CRS for output
stops_final = stops_proj_result.to_crs(tod_stops_gdf.crs)
access_pts_final = access_pts_proj_result.to_crs(access_pts_gdf.crs)

# Summary
print("=== ASSIGNMENT SUMMARY ===")
for label, gdf in [("TOD Stops", stops_final), ("Access Points", access_pts_final)]:
    n = len(gdf)
    n_assigned = (gdf["assignment_status"] == "assigned").sum()
    n_conflict = (gdf["assignment_status"] == "conflict").sum()
    n_no_match = (gdf["assignment_status"] == "no_match").sum()
    print(f"\n{label} ({n} total):")
    print(f"assigned:  {n_assigned}")
    print(f"conflict:  {n_conflict}")
    print(f"no_match:  {n_no_match}")
    if n_conflict + n_no_match > 0:
        print(
            f"  → {n_conflict + n_no_match} rows need review "
            f"(filter assignment_status != 'assigned' in GIS)"
        )


=== ASSIGNMENT SUMMARY ===

TOD Stops (755 total):
assigned:  732
conflict:  22
no_match:  1
  → 23 rows need review (filter assignment_status != 'assigned' in GIS)

Access Points (1182 total):
assigned:  1112
conflict:  66
no_match:  4
  → 70 rows need review (filter assignment_status != 'assigned' in GIS)


In [21]:
# Export 3 development layers to GeoPackage.
# These are in-progress layers — not yet authoritative.  The _dev suffix
# signals that manual review (via REVIEW_XLSX) is still required before
# the final Step 3 output is produced.
print(f"Exporting development layers to GeoPackage: {tod_gtfs_db_gdb_path}")

stations_gdf.to_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_STATIONS_DEV_LAYER, driver="GPKG", mode="w")
print(f"Exported {len(stations_gdf)} stations → '{GPKG_TOD_STATIONS_DEV_LAYER}'")

stops_final.to_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_STOPS_DEV_LAYER, driver="GPKG", mode="w")
print(
    f"Exported {len(stops_final)} TOD stops → '{GPKG_TOD_STOPS_DEV_LAYER}'  "
    f"({(stops_final['assignment_status'] == 'assigned').sum()} assigned, "
    f"{(stops_final['assignment_status'] != 'assigned').sum()} need review)"
)

access_pts_final.to_file(
    tod_gtfs_db_gdb_path, layer=GPKG_TOD_ACCESS_PTS_DEV_LAYER, driver="GPKG", mode="w"
)
print(
    f"Exported {len(access_pts_final)} access points → '{GPKG_TOD_ACCESS_PTS_DEV_LAYER}'  "
    f"({(access_pts_final['assignment_status'] == 'assigned').sum()} assigned, "
    f"{(access_pts_final['assignment_status'] != 'assigned').sum()} need review)"
)

print(f"\n GeoPackage updated: {tod_gtfs_db_gdb_path}")

Exporting development layers to GeoPackage: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg
Exported 450 stations → 'tod_stations_dev'
Exported 755 TOD stops → 'tod_stops_dev'  (732 assigned, 23 need review)
Exported 1182 access points → 'tod_access_points_dev'  (1112 assigned, 70 need review)

 GeoPackage updated: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg


In [22]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

# Columns written to each review sheet
STOPS_REVIEW_COLS = ["stop_id", "stop_name", "tod_tier", "station_id", "assignment_status", "buffer_distance_m"]
ACCESS_REVIEW_COLS = ["access_id", "access_point_name", "agency", "station_id", "assignment_status", "buffer_distance_m"]

# Allowed values for the assignment_status dropdown
STATUS_VALUES = '"conflict,no_match,manual_station_assignment"'

HEADER_FILL = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
HEADER_FONT = Font(bold=True, color="FFFFFF")


def _write_review_sheet(ws, df, cols):
    """Write df columns to ws with a styled header, data validation on
    assignment_status, frozen header row, and auto-fitted column widths."""
    # Write header
    for col_idx, col_name in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center")

    # Write data rows
    df_out = df[[c for c in cols if c in df.columns]].copy()
    # Drop geometry if it snuck in
    for col in df_out.columns:
        if hasattr(df_out[col], "geom_type"):
            df_out = df_out.drop(columns=[col])
    # Convert pandas NA/NaN to None so openpyxl can serialise every value
    df_out = df_out.astype(object).where(df_out.notna(), other=None)
    for row_idx, row in enumerate(df_out.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    # Data validation dropdown on assignment_status column
    status_col_idx = cols.index("assignment_status") + 1
    status_col_letter = get_column_letter(status_col_idx)
    dv = DataValidation(
        type="list",
        formula1=STATUS_VALUES,
        showDropDown=False,  # False = dropdown arrow visible in Excel
        showErrorMessage=True,
        errorTitle="Invalid value",
        error="Please select: conflict, no_match, or manual_station_assignment",
    )
    ws.add_data_validation(dv)
    dv.sqref = f"{status_col_letter}2:{status_col_letter}{len(df_out) + 1}"

    # Freeze header row
    ws.freeze_panes = "A2"

    # Auto-fit column widths
    for col_idx, col_name in enumerate(cols, start=1):
        col_letter = get_column_letter(col_idx)
        max_len = len(col_name)
        for row_idx in range(2, len(df_out) + 2):
            val = ws.cell(row=row_idx, column=col_idx).value
            if val is not None:
                max_len = max(max_len, len(str(val)))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


# Build review DataFrames from the reprojected-back GDFs (no geometry column)
stops_review_df = stops_final.drop(columns=["geometry"], errors="ignore")
access_review_df = access_pts_final.drop(columns=["geometry"], errors="ignore")

# Write workbook
wb = openpyxl.Workbook()
ws_stops = wb.active
ws_stops.title = REVIEW_STOPS_SHEET
_write_review_sheet(ws_stops, stops_review_df, STOPS_REVIEW_COLS)

ws_access = wb.create_sheet(title=REVIEW_ACCESS_PTS_SHEET)
_write_review_sheet(ws_access, access_review_df, ACCESS_REVIEW_COLS)

wb.save(REVIEW_XLSX)

print(f"Review workbook written: {REVIEW_XLSX}")
print(f"   Sheet '{REVIEW_STOPS_SHEET}':       {len(stops_review_df)} rows")
print(f"   Sheet '{REVIEW_ACCESS_PTS_SHEET}': {len(access_review_df)} rows")
print()
print("Next steps:")
print(f"1. Open {REVIEW_XLSX.name}")
print("2. For any row needing a manual station assignment:")
print("   - Set assignment_status = 'manual_station_assignment'")
print("   - Enter the correct station_id")
print("   (Use the dropdown — only the three listed values are valid.)")
print("3. Save the workbook, then run 3_tod_station_assignment_reintegration.ipynb")


Review workbook written: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_review.xlsx
   Sheet 'stops':       755 rows
   Sheet 'access_points': 1182 rows

Next steps:
1. Open SB79_tod_review.xlsx
2. For any row needing a manual station assignment:
   - Set assignment_status = 'manual_station_assignment'
   - Enter the correct station_id
   (Use the dropdown — only the three listed values are valid.)
3. Save the workbook, then run 3_tod_station_assignment_reintegration.ipynb
